**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](../README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

<a id="C01"></a>
**C01**

# CNN Training Studio
Computer Vision · Lezione 12

### Elaborato individuale — da compilare


Modificare questi due campi nella cella di testo. È previsto un solo autore.

Il notebook della lezione 11 è una dimostrazione completa da consultare. Qui si completano i tre blocchi C10, C13 e C17 e i tre richiami **P01** (neurone e passo SGD, C07–C08), **P02** (accumulo dei gradienti, C16–C17) e **P03** (checkpoint, C27). Le risposte N01–N15 sono brevi: bastano 3–5 righe o una piccola tabella; non ricopiare gli output già presenti.

Sono richieste soltanto le due corse A/B già previste. Challenge C36–C37, repliche C38–C39 e ulteriori esercizi D01–D08 della dispensa sono facoltativi e non entrano nella valutazione dell'elaborato. L'discussione conclusiva C42 serve per l'auto-verifica.

Scrivere la sintesi F01–F05 direttamente in C40, entro 500 parole: **non serve una relazione separata**. Consegnare il `.ipynb` con nome, cognome, completamenti, ipotesi, risposte e output, insieme alla cartella `output_lezione12/` compressa. Nome suggerito: `ComputerVision_Lezione12.ipynb`.

Domanda guida: che cosa impara una CNN, che cosa può vedere e come verifichiamo l’effetto di una scelta di training?

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

### Avvio in Colab e salvataggio

Caricare l’intero `CNN_Training_Studio_studenti.zip` nello stesso runtime del notebook; prima di C03 estrarlo in `/content` e scegliere `/content/CNN_Training_Studio_studenti` come cartella corrente. `dati/` e `supporto/` devono restare nella cartella estratta. Usare un runtime **CPU (acceleratore Nessuno)**. Verificare `torch`, `numpy`, `matplotlib` e installare soltanto le librerie mancanti; non serve JupyterLab in Colab. Le istruzioni complete, con celle temporanee di upload e download, sono nel **README dello ZIP**.

C03 mantiene CPU e un thread. Le CNN e gli input sono float32; C08/C15 usano float64 per i controlli analitici. I riferimenti misurati sono locali su CPU: il servizio Colab non è stato provato nella verifica documentata. Registrare le versioni stampate da C03 e annotare Matplotlib.

Dopo un nuovo runtime ripetere il setup e le celle operative in ordine, completando le quattro celle C08, C10, C13 e C17 (P01/P02 inclusi). C20 ricrea gli stati: se la si ripete dopo A, rifare A/B e i blocchi dipendenti. C39 resta disattivata nello studio. Le celle temporanee vanno eliminate, mantenendo i **43 ID C01–C43**.

Dopo C41 usare **File → Scarica → Scarica .ipynb** e scaricare anche l’intera `output_lezione12/`. Conservare ipotesi, risposte P03/N01–N15 e sintesi F01–F05 nel notebook: non sono tutte esportate automaticamente nei JSON. La diagnosi di una challenge si aggiunge soltanto se si sceglie l’approfondimento facoltativo.

<a id="C02"></a>
**C02**

## Percorso individuale e lettura nello studio

### Lessico per leggere la rete

**CNN** significa *Convolutional Neural Network*, rete neurale convoluzionale. Un **neurone affine** somma ingressi moltiplicati per pesi e aggiunge un bias: $a=\sum_i w_i x_i+b$. **ReLU** (*Rectified Linear Unit*) calcola $h=\max(0, a)$: per esempio, −2 diventa 0 e 2 resta 2. Un filtro convoluzionale applica una somma pesata a una piccola finestra (**patch**) e riusa gli stessi pesi (**kernel**) nelle diverse posizioni. Le risposte formano una **feature map**, una griglia che non rappresenta automaticamente una classe. Il toy di C08 isola la parte lineare senza bias né ReLU.

Un **canale** è una griglia di valori. **NCHW** significa numero di esempi del batch, canali, altezza e larghezza; N è indicato anche con B. **BN** significa *Batch Normalization* e normalizza le attivazioni separatamente per canale; `Identity` lascia il proprio ingresso invariato. Il **max pooling** conserva il massimo di una finestra; con finestra 2×2 e passo 2 dimezza il lato. `Flatten(1)` appiattisce le dimensioni delle feature conservando il batch. La testa produce dieci **score s**, detti **logits** per immagine; il batch S ha forma B×10 e non contiene ancora probabilità.

I **parametri** sono i tensori aggiornati dall’ottimizzatore. I **buffer** sono altri tensori registrati nello stato del modello: in BN conservano media, varianza e contatore dei batch; il forward in train può modificarli anche senza backward. Lo **stato del modello** comprende parametri e buffer.

I **Checkpoint 1–4** della percorso sono controlli di comprensione; un **checkpoint del modello** è invece una copia indipendente dello stato, inclusi i buffer BN.

Completare in autonomia tre blocchi: C10 architettura, C13 receptive field, C17 modalità. I numeri Cxx includono le celle di testo e restano identici nelle versioni. Nello studio si presentano e si discutono i quattro blocchi: C03–13, C15–17, C20–28, C30–35. Eseguire in ordine. Il test è escluso. Gli output preparati in `recupero/` sono misure dell’archivio originale, da dichiarare quando usati. Installare prima dell’esecuzione i pacchetti indicati in requirements.txt. 

Le aggiunte P01–P03 sono distribuite nei blocchi pertinenti e si completano individualmente in autonomia. I riferimenti Cxx restano gli stessi e gli output di riferimento permettono confronti esplicitamente dichiarati.

In [ ]:
# C03
from pathlib import Path
import sys, copy, json, hashlib, platform
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dati/digits_cv2026.npz").exists())
sys.path.insert(0, str(ROOT / "supporto"))
import cnn_utils as u
SEED = 20261022
u.configure(SEED)
OUT = Path.cwd() / "output_lezione12"
OUT.mkdir(exist_ok=True)
print({"Python":platform.python_version(),"PyTorch":torch.__version__,"NumPy":np.__version__,"device":"cpu","threads":torch.get_num_threads(),"seed":SEED})

<a id="C04"></a>
**C04**

## N01 · Dati e preprocessing
**dtype** indica il tipo numerico di un tensore: float32 e float64 usano rispettivamente 32 e 64 bit per valore. Le immagini della CNN usano float32; i piccoli controlli analitici usano float64 per ridurre gli effetti dell’arrotondamento. La **shape** è la sequenza delle dimensioni; NCHW è stata definita in C02.

Verificare split, dtype e NCHW. Perché usiamo una media/deviazione di canale e non le 64 statistiche della L10? Quali immagini entrano nel fit delle statistiche?

**Risposta N01 (3–5 righe o piccola tabella):** [scrivere qui; richiamare gli output senza ricopiarli]

In [ ]:
# C05
path = ROOT / "dati/digits_cv2026.npz"
data_hash = hashlib.sha256(path.read_bytes()).hexdigest()
z = np.load(path)
print("Chiavi:", z.files)
images, targets = z["images"], z["target"]
indices = {"train": z["train_idx"], "val": z["val_idx"]}
scaled = images.astype(np.float64) / 16.0
mu, sigma = float(scaled[indices["train"]].mean()), float(scaled[indices["train"]].std(ddof=0))
assert sigma > 0
x_all = torch.from_numpy(((scaled-mu)/sigma).astype(np.float32)).unsqueeze(1)
y_all = torch.from_numpy(targets.astype(np.int64))
ds = {k:TensorDataset(x_all[ix], y_all[ix], torch.from_numpy(ix.astype(np.int64))) for k,ix in indices.items()}
print("Media/std train:", mu,sigma,"; SHA256:",data_hash)
print({k:(len(v),tuple(v.tensors[0].shape),str(v.tensors[0].dtype),str(v.tensors[1].dtype)) for k,v in ds.items()})
assert set(indices["train"]).isdisjoint(indices["val"])
assert len(ds["train"]) == 1077 and len(ds["val"]) == 360

In [ ]:
# C06
u.data_fig(images,targets,indices["train"],OUT)

<a id="C07"></a>
**C07**

## N02 · Gradiente di un kernel condiviso

### P01 · Dal neurone al peso condiviso

Riprendere il neurone mostrato nella lezione 11 con dati diversi: $x=2$, $w=2$, $b=-1$, target $y=1$. Usare $a=wx+b$, $h=\operatorname{ReLU}(a)$ e $L=(h-y)^2/2$. In C08 completare il breve blocco P01: forward, backward e **un solo passo SGD simultaneo** per w e b con $\eta=0.1$. Leggere anche la derivata rispetto a x, che rimane un dato e non viene aggiornato. I gradienti si calcolano sui parametri precedenti; backward, da solo, non li modifica.

Confrontare loss prima/dopo e spiegare in 2–3 righe perché aggiornare w e poi ricalcolare il gradiente di b produrrebbe un'altra procedura. Questo è un calcolo scalare, distinto dalla CNN e dal suo ottimizzatore Adam.

**Commento P01:** [scrivere qui]


**Lettura del grafo.** Il forward costruisce le quattro uscite dalle quattro patch, poi la loss. Il gradiente in arrivo (**upstream**) è $\delta_{ij}=\partial L/\partial Y_{ij}$. Per questa loss quadratica vale $\delta=Y$. Ogni uso dello stesso peso contribuisce alla sua derivata: moltiplicare δ per il pixel pertinente e sommare sui percorsi. Per K00 seguire prima le quattro patch una alla volta; poi ripetere per gli altri pesi. Lo step dell’ottimizzatore viene dopo il calcolo e la verifica del gradiente.

**Controlli della L11.** Autograd calcola le derivate seguendo il grafo delle operazioni. La differenza centrale perturba un solo peso di +ε e −ε, lasciando gli altri fissi, e confronta le due loss: ε è il passo del controllo numerico, non il learning rate. La cross-correlazione usa il kernel senza ribaltarlo; una finestra valida resta interamente dentro l’ingresso, senza padding.

Calcolare Y e ∂L/∂K prima di eseguire. Loss = 0.5 somma(Y²); nessuna media. Spiegare perché contribuiscono tutte le patch. Verificare autograd, quattro differenze centrali e uno step SGD con η=0.1.

**Risposta N02 (3–5 righe o piccola tabella):** [scrivere qui; richiamare gli output senza ricopiarli]

In [ ]:
# C08

# P01 — completare questo richiamo prima di leggere il kernel condiviso.
x_p01=torch.tensor(2.,dtype=torch.float64,requires_grad=True)
w_p01=torch.tensor(2.,dtype=torch.float64,requires_grad=True)
b_p01=torch.tensor(-1.,dtype=torch.float64,requires_grad=True)
a_p01=... # TODO P01: somma pesata e bias
h_p01=... # TODO P01: attivazione ReLU
loss_p01=... # TODO P01: loss quadratica con target 1
... # TODO P01: calcolare i gradienti
gw_p01=w_p01.grad.detach().clone();gb_p01=b_p01.grad.detach().clone()
w_new_p01=... # TODO P01: passo SGD dal vecchio w
b_new_p01=... # TODO P01: passo SGD dal vecchio b
new_loss_p01=.5*(torch.relu(w_new_p01*x_p01+b_new_p01)-1).square()
p01_result={"a":a_p01.item(),"h":h_p01.item(),"loss_before":loss_p01.item(),"dw":gw_p01.item(),"db":gb_p01.item(),"dx":x_p01.grad.item(),"w_after":w_new_p01.item(),"b_after":b_new_p01.item(),"loss_after":new_loss_p01.item(),"backward_did_not_update":w_p01.item()==2. and b_p01.item()==-1.}
print("P01",json.dumps(p01_result,indent=2))

X=torch.tensor([[[[1.,2.,0.],[0.,1.,3.],[2.,1.,0.]]]],dtype=torch.float64)
K=torch.tensor([[[[1.,0.],[0.,-1.]]]],dtype=torch.float64,requires_grad=True)
f=lambda k: .5*nn.functional.conv2d(X,k).square().sum()
Y=nn.functional.conv2d(X,K);L=f(K);L.backward();g=K.grad.clone();fd=torch.zeros_like(K)
for a in range(2):
    for b in range(2):
        e=torch.zeros_like(K);e[0,0,a,b]=1e-5
        fd[0,0,a,b]=(f(K.detach()+e)-f(K.detach()-e))/(2e-5)
assert torch.allclose(g,fd,atol=1e-8,rtol=1e-8)
Knew=K.detach()-.1*g
kernel_result={"Y":Y.detach().tolist(),"loss":float(L.detach()),"grad":g.tolist(),"new_K":Knew.tolist(),"new_loss":float(f(Knew)),"finite_difference_max_error":float((fd-g).abs().max())}
print(json.dumps(kernel_result,indent=2))

<a id="C09"></a>
**C09**

## N03 · Architettura

**Operazioni.** `Conv` applica filtri condivisi; `ReLU` applica max(0, a); `BN` indica Batch Normalization. `Pool` è il max pooling; `Flatten` appiattisce le feature conservando B; `Linear` è una trasformazione affine che restituisce gli score. **Stride** è il passo della finestra sul proprio ingresso; **padding** tratta il bordo attorno all’ingresso: qui le Conv usano zeri e il pooling non aggiunge bordo.

**Pool e Flatten.** In ogni finestra 2×2 il max pooling conserva il massimo; le quattro finestre non sovrapposte di una mappa 4×4 producono una mappa 2×2. Non introduce pesi appresi. `Flatten(1)` riordina le feature in un vettore per esempio: B×16×2×2 diventa B×64, senza mescolare esempi. **Padding 1**, nelle Conv qui usate, aggiunge un pixel di zeri su ciascun lato; il pooling ha padding 0. **Stride 1** sposta la finestra di un pixel sul suo ingresso. Il simbolo t denota lo stride nelle formule.

**Altre opzioni del contratto.** Dilation indica la distanza fra posizioni campionate dal kernel: con valore 1 sono contigue. `groups=1` collega ogni filtro a tutti i canali in ingresso. `ceil_mode=False` usa il conteggio per difetto delle finestre complete del pooling.

Completare Conv/BN/Pool/Flatten. Due blocchi 3×3, padding1, bias=False, canali1→8→16. BN in A, Identity in B. Testa64→10. Calcolare forme, parametri e buffer separatamente.

**Risposta N03 (3–5 righe o piccola tabella):** [scrivere qui; richiamare gli output senza ricopiarli]

In [ ]:
# C10
class SmallCNN(nn.Module):
    def __init__(self,use_bn=True):
        super().__init__()
        self.conv1=... # TODO: Conv1
        self.bn1=nn.BatchNorm2d(8) if use_bn else nn.Identity()
        self.relu1=nn.ReLU();self.pool1=nn.MaxPool2d(2,2)
        self.conv2=... # TODO: Conv2
        self.bn2=nn.BatchNorm2d(16) if use_bn else nn.Identity()
        self.relu2=nn.ReLU();self.pool2=nn.MaxPool2d(2,2)
        self.flatten=... # TODO: conservare asse batch
        self.head=... # TODO: logits
    def forward(self,x):
        x=self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x=self.pool2(self.relu2(self.bn2(self.conv2(x))))
        return self.head(self.flatten(x))

In [ ]:
# C11
a0,b0=u.init_pair(SmallCNN,SEED)
shape_rows=[]
x=torch.zeros(2,1,8,8)
with torch.no_grad():
    a0.eval()
    for name,layer in a0.named_children():
        x=layer(x);shape_rows.append((name,tuple(x.shape),sum(p.numel() for p in layer.parameters())))
print(*shape_rows,sep="\n")
print("Parametri A/B:",*[sum(p.numel() for p in m.parameters()) for m in [a0,b0]])
print("Buffer A:",[(n,tuple(t.shape),t.numel()) for n,t in a0.named_buffers()])
assert sum(p.numel() for p in a0.parameters())==1922
assert sum(p.numel() for p in b0.parameters())==1874

<a id="C12"></a>
**C12**

## N04 · Receptive field

**RF** significa *receptive field*, campo recettivo: le posizioni dell’input da cui un’uscita può dipendere. r misura l’ampiezza nominale in pixel dell’input; j la distanza fra i centri di uscite adiacenti, sempre riferita all’input. Lo stride è indicato con t nelle formule, riservando s agli score della testa; la variabile locale `s` in C13 rappresenta t. Non confondere dimensione del kernel, passo del layer e supporto accumulato.

**Supporto e dipendenza effettiva.** Il campo recettivo nominale riguarda i possibili collegamenti Conv/Pool; pesi, ReLU e max pooling determinano quali contribuiscono al singolo output. Con BN in eval le statistiche sono fissate; in train sono condivise fra esempi e posizioni dello stesso canale e aggiungono dipendenze oltre il supporto locale.

Compilare r e j per Conv3/Pool2/Conv3/Pool2. Il supporto nominale include padding: non è un numero di pixel osservati. Specificare la differenza tra BN in eval con statistiche fisse e BN in train.

**Risposta N04 (3–5 righe o piccola tabella):** [scrivere qui; richiamare gli output senza ricopiarli]

In [ ]:
# C13
r,j=1,1
rf_rows=[]
for name,k,s in [("Conv1",3,1),("Pool1",2,2),("Conv2",3,1),("Pool2",2,2)]:
    r=... # TODO: usa il passo precedente
    j=... # TODO: nuovo passo
    rf_rows.append((name,r,j))
print(rf_rows)
assert (r,j)==(10,4)

<a id="C14"></a>
**C14**

## N05 · BatchNorm su quattro valori

**Lettura per passaggi.** BN raccoglie i valori di uno stesso canale sui batch e sulle posizioni. Prima centra: sottrae la media; poi scala: divide per la radice della varianza più ε. Infine applica la scala appresa γ e la traslazione appresa β. Nel toy leggere separatamente (1) il calcolo delle statistiche del batch, (2) il forward normalizzato, (3) l’aggiornamento delle statistiche mobili e (4) il successivo pass in eval.

**Statistiche e stato.** m è il numero di valori raccolti per canale: N×H×W. La varianza del forward usa il divisore m; la stima che entra in `running_var` usa m−1, con m>1. Le statistiche mobili sono memorie aggiornate come `(1−α)·vecchia + α·batch`, con α=`momentum`. Questo coefficiente BN non è il momentum dell’ottimizzatore della L11. Gamma e beta sono parametri; `running_mean`, `running_var` e il contatore sono buffer. ε stabilizza la divisione ed è **dentro** la radice `sqrt(var+ε)`, diversamente dall’epsilon di Adam.

Calcolare media, varianza del forward, varianza che aggiorna il buffer, running_mean e running_var. Prevedere l’output in eval dopo questo batch. γ=1, β=0, eps=1e−5, momentum=0.1.

**Risposta N05 (3–5 righe o piccola tabella):** [scrivere qui; richiamare gli output senza ricopiarli]

In [ ]:
# C15
bn=nn.BatchNorm2d(1,dtype=torch.float64)
v=torch.tensor([1.,3.,5.,7.],dtype=torch.float64).reshape(2,1,1,2)
with torch.no_grad():
    train_out=bn(v);bn.eval();eval_out=bn(v)
bn_result={"mean":float(v.mean()),"variance_forward":float(v.var(unbiased=False)),"variance_buffer_input":float(v.var(unbiased=True)),"running_mean":float(bn.running_mean.item()),"running_var":float(bn.running_var.item()),"train_output":train_out.flatten().tolist(),"eval_after_train":eval_out.flatten().tolist()}
print(json.dumps(bn_result,indent=2))

<a id="C16"></a>
**C16**

## N06 · Modalità e stato
**Due controlli distinti.** `train()` e `eval()` impostano la modalità dei moduli: BN in train usa il batch e aggiorna le statistiche mobili; in eval usa quelle conservate. `no_grad()` disattiva il tracciamento di autograd nel blocco, ma non cambia modalità, non azzera gradienti già presenti e non impedisce gli aggiornamenti dei buffer in train. `zero_grad()` azzera i gradienti accumulati; `step()` aggiorna i parametri.

Due copie dello stesso stato, stesso batch training. Quali elementi cambiano con train+no_grad? Completare eval per la seconda copia. Non usare queste copie nelle corse A/B. N12: perché il checkpoint deve contenere i buffer?

### P02 · Accumulo e azzeramento dei gradienti

In fondo a C17 usare un parametro scalare separato $\theta=2$ e $L=\theta^2/2$. Eseguire due forward/backward distinti **senza step e senza azzerare tra i primi due**; poi azzerare con l'ottimizzatore e fare un terzo forward/backward. Completare le tre righe indicate, senza usare `retain_graph` e senza modificare A, B o le loro copie.

Prevedere i tre gradienti prima di eseguire. Spiegare in 2–3 righe perché il parametro resta 2 e perché `no_grad()` non sostituisce `zero_grad()`.

**Commento P02:** [scrivere qui]


**Risposta N06 (3–5 righe o piccola tabella):** [scrivere qui; richiamare gli output senza ricopiarli]

In [ ]:
# C17
train_copy=copy.deepcopy(a0).train();eval_copy=copy.deepcopy(a0).train()
# Entrambe partono in train: il cambio di modalità qui sotto deve essere esplicito.
xb=ds["train"].tensors[0][:64]
before_train=u.clone_state(train_copy);before_eval=u.clone_state(eval_copy)
... # TODO: imposta la modalità di valutazione
with torch.no_grad():
    yt=train_copy(xb);ye=eval_copy(xb)
mode_result={"changed_train_keys":[k for k in before_train if not torch.equal(before_train[k],train_copy.state_dict()[k])],"eval_state_unchanged":u.same_state(before_eval,u.clone_state(eval_copy)),"max_output_difference":float((yt-ye).abs().max())}
print(json.dumps(mode_result,indent=2))
assert mode_result["eval_state_unchanged"], "La copia di valutazione modifica i buffer: completa il cambio di modalità in C17"

# P02 — esempio indipendente: non usa le reti A/B e non fa alcuno step.
theta_p02=torch.tensor(2.,dtype=torch.float64,requires_grad=True)
opt_p02=torch.optim.SGD([theta_p02],lr=.1)
(.5*theta_p02.square()).backward()
g_first_p02=theta_p02.grad.item()
second_loss_p02=.5*theta_p02.square() # nuovo grafo, stesso parametro
... # TODO P02: secondo backward senza azzerare
g_accumulated_p02=theta_p02.grad.item()
... # TODO P02: azzerare i gradienti, senza step
third_loss_p02=.5*theta_p02.square() # altro nuovo grafo
... # TODO P02: terzo backward sul nuovo grafo
p02_result={"gradient_first":g_first_p02,"gradient_after_two":g_accumulated_p02,"gradient_after_reset":theta_p02.grad.item(),"parameter_without_step":theta_p02.item()}
print("P02",json.dumps(p02_result,indent=2))


<a id="C18"></a>
**C18**

### Checkpoint 2
Spiegare modalità, autograd, parametri e buffer. Salvare il notebook prima delle corse. Il blocco successivo ricrea reti, ottimizzatori e loader ufficiali. 

<a id="C19"></a>
**C19**

## N07 · Pairing
**Pairing** significa appaiamento: A e B partono dagli stessi pesi condivisi e incontrano gli stessi esempi nello stesso ordine. Il **seed** inizializza un generatore pseudocasuale; il **loader** fornisce in sequenza i mini-batch. Leggerlo può avanzarne il generatore: per il controllo usiamo copie sacrificabili, non l’iteratore ufficiale. Un **hash** è un’impronta del contenuto, utile a confrontare dati o sequenze senza stamparli interamente; non è una misura della qualità del modello.

Confrontare tensori comuni iniziali e ID dei batch; un seed scritto nel file non è una verifica sufficiente. L’ispezione usa loader sacrificabili: il training ne costruisce nuovi.

**Risposta N07 (3–5 righe o piccola tabella):** [scrivere qui; richiamare gli output senza ricopiarli]

In [ ]:
# C20
A,B=u.init_pair(SmallCNN,SEED)
initial_A=copy.deepcopy(A);initial_B=copy.deepcopy(B)
common_equal=all(torch.equal(A.state_dict()[k],v) for k,v in B.state_dict().items())
ids_a=[ids.tolist() for _,_,ids in u.loader(ds["train"],SEED,True)]
ids_b=[ids.tolist() for _,_,ids in u.loader(ds["train"],SEED,True)]
assert common_equal and ids_a==ids_b
print("Pesi comuni uguali:",common_equal,"Batch identici:",ids_a==ids_b,"batch/ultimo:",len(ids_a),len(ids_a[-1]))

<a id="C21"></a>
**C21**

## N08 · Piccolo fit di controllo
Su una copia B: 20 immagini training, due per classe con minori ID, 200 update Adam. Che cosa verifica un forte calo della loss? Che cosa non dimostra? La copia viene scartata.

**Risposta N08 (3–5 righe o piccola tabella):** [scrivere qui; richiamare gli output senza ricopiarli]

In [ ]:
# C22
tiny=u.tiny_fit(initial_B,ds["train"]);print(json.dumps(tiny,indent=2));u.plot_tiny(tiny,OUT)

<a id="C23"></a>
**C23**

## N09 · Sonde
Una **sonda** registra una misura in un punto del calcolo, senza aggiungere un nuovo esperimento. La **norma euclidea** di un tensore è la radice della somma dei quadrati dei suoi elementi: riassume una grandezza complessiva, non quella di una sola coordinata. **Δθ=θ_dopo−θ_prima** è l’aggiornamento effettivo; il gradiente è misurato dopo backward e prima di step. La frazione di zeri è il numero di attivazioni nulle diviso per il loro numero totale.

Le sonde usano il forward reale del primo batch alle epoche 1/30: Conv/ReLU, frazione di zeri, norma gradiente, norma Δparametro. Nessun forward aggiuntivo in train. Per Adam η‖g‖ non è in generale ‖Δθ‖. Il rapporto è non definito quando la norma iniziale è zero.

**Risposta N09 (3–5 righe o piccola tabella):** [scrivere qui; richiamare gli output senza ricopiarli]

<a id="C24"></a>
**C24**

## N10 · Baseline A

**Epoca e aggiornamento.** Un’epoca è un passaggio completo sui 1077 esempi training, divisi in mini-batch. Un aggiornamento è uno `Adam.step()` dopo il backward su un mini-batch. Con ultimo batch incluso: 16×64+53=1077, dunque 17 aggiornamenti per epoca e 510 in 30 epoche. Il budget fisso completa tutte le epoche; la selezione del checkpoint non interrompe il fit e quindi non è early stopping.

**Tre misure, prima di leggere le curve.** La CE (*cross-entropy*) riceve gli score s, non probabilità già trasformate. `online_loss` aggrega le loss osservate durante i passi in train, mentre i pesi cambiano. `train_eval` misura il training a fine epoca in eval, con stato fissato; `validation` misura nello stesso modo lo split separato. Il gap qui è **CE_validation−CE_train_eval**, non la differenza con `online_loss`. **Macro-F1**, già incontrata nella L9, è la media degli F1 calcolati separatamente sulle dieci classi; la confusion matrix si accumula sullo split intero.

30 epoche, Adam lr 0.001, batch 64, nessuna augmentation/dropout/weight decay. Osservare online_loss, train_eval e validation. Selezione: massimo macro-F1 validation, prima epoca a parità esatta. Il test resta escluso.

**Risposta N10 (3–5 righe o piccola tabella):** [scrivere qui; richiamare gli output senza ricopiarli]

In [ ]:
# C25
A=copy.deepcopy(initial_A)
result_A=u.run(A,ds,SEED)
print("A checkpoint",result_A["best_epoch"],u.brief(result_A["best_val"]),"secondi",result_A["seconds_training_and_eval"])

In [ ]:
# C26
print("Sonda primo batch:",json.dumps(result_A["probes"][0],indent=2))
print("Ultima epoca:",result_A["last"])

<a id="C27"></a>
**C27**

## N12 · Checkpoint e valutazione

**Copia, ripristino e report.** Un checkpoint conserva una copia indipendente dello stato del modello, inclusi i buffer BN. La copia viene aggiornata durante il training solo per macro-F1 validation strettamente maggiore; la prima epoca vince le parità. Ripristinare significa ricaricare quella copia dopo il budget per valutare e confrontare lo stesso stato selezionato. Il report è la relazione che descrive protocollo, checkpoint, misure, figure e limiti. Qui il test resta escluso: il report usa le evidenze previste sulla validation.

**Copia indipendente.** Conservare soltanto un riferimento a `state_dict()` non congela i tensori: occorre una copia indipendente, oppure un salvataggio da ricaricare. Per confrontare i logits in eval servono architettura, preprocessing e stato completo; per riprendere lo stesso training servono anche stato dell’ottimizzatore e dei generatori.

Verificare che eval lasci intatti parametri e buffer, ricaricare una copia e confrontare i logits. Perché scegliere un checkpoint non equivale a fermare il training?

### P03 · Quale checkpoint conservare?

Questi sono **dati sintetici per un mini-esempio di quattro epoche**, non risultati delle corse A/B. La regola massimizza macro-F1 validation, conserva la prima epoca nelle parità esatte e completa il budget fissato.

| Epoca | CE validation | Macro-F1 validation |
|---:|---:|---:|
| 1 | 0.90 | 0.70 |
| 2 | 0.70 | 0.75 |
| 3 | 0.60 | 0.75 |
| 4 | 0.50 | 0.74 |

Indicare l'epoca conservata, quante epoche si eseguono e quale stato si ricarica prima del report. Spiegare perché il minimo di CE non decide la scelta e perché non si tratta di early stopping. Il test si usa in questa consegna? Per le corse reali il budget rimane **30 epoche**, non quattro.

**Risposta P03 (3–5 righe):** [scrivere qui]

La risposta P03 copre già la scelta del checkpoint in N12: per N12 aggiungere soltanto l'esito della verifica di ricarica in C28, senza riscrivere la stessa spiegazione.


**Risposta N12 (3–5 righe o piccola tabella):** [scrivere qui; richiamare gli output senza ricopiarli]

In [ ]:
# C28
before=u.clone_state(A)
check=u.evaluate(A,ds["val"])
clone=SmallCNN(True);clone.load_state_dict(result_A["best_state"])
reloaded=u.evaluate(clone,ds["val"])
assert u.same_state(before,u.clone_state(A))
assert np.array_equal(check["logits"],reloaded["logits"])
print("Stato invariato e logits ricaricati identici: PASS")

<a id="C29"></a>
**C29**

## N11 · Ipotesi prima di B
Scrivere la direzione attesa per Δmacro-F1=B−A, unica modifica e limite. Si rimuovono entrambe le BN, inclusi48 parametri affine; le Conv conservano bias=False. Tutti gli altri fattori restano fissi.

<a id="C30"></a>
**C30**

**Ipotesi individuale, prima di eseguire C31:**

[Scrivere qui previsione, misura primaria e condizioni. Non cambiare configurazione dopo il risultato.]

Se hai già visto i risultati delle figure o dei log di riferimento, dichiaralo qui: non presentare come previsione indipendente una spiegazione formulata dopo. Il confronto resta una replica didattica con protocollo fissato. Registrare la previsione prima di osservare i risultati di riferimento e completare autonomamente il codice.

In [ ]:
# C31
B=copy.deepcopy(initial_B)
result_B=u.run(B,ds,SEED)
assert result_A["batch_hashes"]==result_B["batch_hashes"]
print("B checkpoint",result_B["best_epoch"],u.brief(result_B["best_val"]),"secondi",result_B["seconds_training_and_eval"])

<a id="C32"></a>
**C32**

## N13 · Confronto
Un’**ablation** rimuove o sostituisce un componente nel confronto controllato. Qui B sostituisce BN con Identity: l’uscita del modulo diventa identica al suo ingresso e si rimuovono anche i parametri affine di BN. **Δmacro-F1=B−A** fissa il verso del confronto: una differenza negativa favorisce A nella misura osservata, senza dimostrare una legge generale.

Confrontare checkpoint e ultima epoca, gap di loss, macro-F1 e costo. Un miglioramento in un seed non è una legge generale; assenza di differenza non è equivalenza. Il confronto diretto con L10 include anche un preprocessing diverso.

**Risposta N13 (3–5 righe o piccola tabella):** [scrivere qui; richiamare gli output senza ricopiarli]

In [ ]:
# C33
summary={k:{"best_epoch":r["best_epoch"],"train_eval":u.brief(r["best_train"]),"validation":u.brief(r["best_val"]),"gap_loss":r["best_val"]["loss"]-r["best_train"]["loss"],"last":r["last"],"seconds":r["seconds_training_and_eval"],"steps":r["steps"]}for k,r in [("A",result_A),("B",result_B)]}
summary["delta_macro_f1_B_minus_A"]=result_B["best_val"]["macro_f1"]-result_A["best_val"]["macro_f1"]
print(json.dumps(summary,indent=2));u.plots(result_A,result_B,OUT)

<a id="C34"></a>
**C34**

## N14 · Evidenza visiva
**Scala comune** significa che lo stesso colore corrisponde allo stesso valore nei due pannelli: consente di confrontare prima e dopo senza che un riscalamento separato amplifichi le differenze. Un kernel visualizza pesi; una feature map visualizza risposte a un dato input. Nessuno dei due è automaticamente una mappa di importanza causale.

Stesso input training, kernel e feature map prima/dopo con scale comuni, modalità eval. Descrivere pattern osservati senza inventare ruoli semantici dei canali. Tre errori validation con minori ID: proporre un controllo, senza modificare retroattivamente il protocollo.

**Risposta N14 (3–5 righe o piccola tabella):** [scrivere qui; richiamare gli output senza ricopiarli]

In [ ]:
# C35
u.visual(initial_A,A,images,ds,result_A,OUT)

<a id="C36"></a>
**C36**

## C01–C03 · Approfondimento facoltativo, non valutato
C01: validation in train dentro no_grad. C02: pesi e loader non appaiati. C03: loss online/train confrontata con eval/validation. Prima prevedere il bug; poi eseguire, scrivere osservazione, controllo, correzione e limite. Gli esperimenti lavorano su copie.

Queste diagnosi sono discusse nello studio. Per l’elaborato individuale non è richiesta una challenge: C37 è già fornita e può essere saltata senza impedire C41. Se la si approfondisce, scegliere un solo caso e conservarne il commento; non sostituisce P01–P03 o i completamenti principali.

In [ ]:
# C37
choice="C01" # scegliere una sola challenge
challenge_result=u.challenges(choice,A,ds,SmallCNN,SEED)
print(json.dumps(challenge_result,indent=2))

<a id="C38"></a>
**C38**

## N16 · Repliche indipendenti
**Replica appaiata.** Per ogni seed si rifà la coppia A/B con i fattori comuni fissati. La media riassume le differenze B−A; la deviazione campionaria ne descrive la dispersione, usando il divisore n−1 per n repliche. Tre differenze non bastano, da sole, a stabilire significatività o equivalenza.

Estensione facoltativa e non valutata: non è richiesta per l’elaborato e `RUN_REPLICATES` resta False. Se svolta per interesse personale, usa tre seed appaiati 20261022/23/24, tutti riportati, nessuna scelta del migliore. La media e la deviazione campionaria descrivono questi tre valori; non dimostrano significatività.

In [ ]:
# C39
RUN_REPLICATES=False
if RUN_REPLICATES:
    deltas=[]
    for seed in [20261022,20261023,20261024]:
        ar,br=u.init_pair(SmallCNN,seed);ra=u.run(ar,ds,seed);rb=u.run(br,ds,seed)
        d=rb["best_val"]["macro_f1"]-ra["best_val"]["macro_f1"];deltas.append(d)
        print(seed,ra["best_val"]["macro_f1"],rb["best_val"]["macro_f1"],d)
    print("Media delta:",np.mean(deltas),"std campionaria:",np.std(deltas,ddof=1))

<a id="C40"></a>
**C40**

## N15 / F01–F05 · Sintesi individuale per la consegna

Completare qui una sintesi di **massimo 500 parole** . Non è richiesta una relazione separata; richiamare celle, tabelle e figure già presenti senza ricopiarle. Le risposte P01–P03 restano nei rispettivi blocchi.

**F01 — Ipotesi e protocollo:** [richiamare l’ipotesi registrata in C30 prima di B; unica modifica e fattori fissi]

**F02 — Architettura e controlli:** [forme/conteggi/RF essenziali e controlli effettivamente superati; citare le celle]

**F03 — Misure e curve:** [checkpoint A/B, differenza macro-F1 B−A e gap dichiarato; richiamare tabella e grafici C33]

**F04 — Interpretazione e limite:** [una osservazione documentata, spiegazione possibile e controllo; nessuna challenge aggiuntiva obbligatoria]

**F05 — Riproducibilità e provenienza:** [ambiente, seed, stato/ricarica e origine dei log; indicare eventuali parti eseguite con risultati di recupero]

Prima di salvare, controllare codice completato, ipotesi C30, risposte brevi e output. Dopo C41 salvare il notebook `.ipynb` e comprimere `output_lezione12/`. Il test resta escluso. Non è richiesta una soglia minima di accuracy né un risultato favorevole a BN. Se si usano i log di recupero, dichiarare esattamente quali parti sono proprie e quali fornite nell’archivio originale.

In [ ]:
# C41
u.save_run("A",result_A,OUT,mu,sigma,SEED,data_hash,ds)
u.save_run("B",result_B,OUT,mu,sigma,SEED,data_hash,ds)
loaded=torch.load(OUT/"checkpoint_A.pt",map_location="cpu",weights_only=True)
reload=SmallCNN(True);reload.load_state_dict(loaded["state_dict"])
assert np.array_equal(u.evaluate(reload,ds["val"])["logits"],result_A["best_val"]["logits"])
metadata={"esercizi":{"P01":p01_result,"P02":p02_result},"summary":summary,"input_mean":mu,"input_std":sigma,"kernel":kernel_result,"batchnorm":bn_result,"mode":mode_result,"tiny_fit":tiny,"environment":{"python":platform.python_version(),"numpy":np.__version__,"torch":torch.__version__,"platform":platform.platform(),"threads":torch.get_num_threads()},"data_sha256":data_hash,"test_evaluated":False,"checkpoint_reload_identical":True}
(OUT/"metriche_finali.json").write_text(json.dumps(metadata,indent=2))
print("Esportati A/B, configurazione e log. Ricarica identica: PASS")

<a id="C42"></a>
**C42**

## Auto-verifica E01–E05
1. Come il gradiente di un peso incorpora più posizioni?
2. r e j dopo Conv 3×3 stride 1 e Pool 2×2 stride 2?
3. Perché no_grad non basta con BN?
4. Tre controlli che rendono A/B appaiato?
5. Quale conclusione se Δmacro-F1 è vicino a zero in un seed?

Queste cinque domande ricapitolano il percorso e sono discusse nello studio: non occorre aggiungere cinque risposte alla consegna. Le risposte già sviluppate nei blocchi precedenti sono sufficienti.

<a id="C43"></a>
**C43**

## Fonti operative
[PyTorch Conv2d](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Conv2d.html), [BatchNorm2d](https://docs.pytorch.org/docs/2.14/generated/torch.nn.BatchNorm2d.html), [Adam](https://docs.pytorch.org/docs/2.14/generated/torch.optim.Adam.html). Dati: Digits scikit-learn, copia locale e split della lezione9. Teoria: libro Visione computazionale, materiali L3/L11, D2L e CS231n, riferimenti completi nella dispensa.